In [1]:
#load packages
import numpy as np
import xarray as xr
import math
import csv

import matplotlib.pyplot as plt
%matplotlib inline

import os
import pandas as pd
import cmocean
import matplotlib.gridspec as gridspec

from scipy.stats import norm
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, zoomed_inset_axes

# import gsw_xarray as gsw_xr # seawater calculations - might not need this one
import gsw as gsw
## mapping packages
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from pyproj import Transformer, Geod
from shapely.geometry import LineString, Point
from scipy.signal import savgol_filter
from scipy.interpolate import griddata

In [2]:
cd /g/data/jk72/deg581/seqom/analysis/notebooks

/g/data/jk72/deg581/seqom/analysis/notebooks


In [3]:
#define functions
def inpolygon(xq, yq, xv, yv):
    from matplotlib import path
    shape = xq.shape
    xq = xq.reshape(-1)
    yq = yq.reshape(-1)
    xv = xv.reshape(-1)
    yv = yv.reshape(-1)
    q = [(xq[i], yq[i]) for i in range(xq.shape[0])]
    p = path.Path([(xv[i], yv[i]) for i in range(xv.shape[0])])
    return p.contains_points(q).reshape(shape)

from xgcm import Grid

# map u,v to rho points
def ROMSmetricsAndGrid(ds):
    ds = ds.rename({'eta_u': 'eta_rho', 'xi_v': 'xi_rho', 'xi_psi': 'xi_u', 'eta_psi': 'eta_v'})

    coords={'X':{'center':'xi_rho', 'inner':'xi_u'}, 
        'Y':{'center':'eta_rho', 'inner':'eta_v'}, 
        'Z':{'center':'s_rho', 'outer':'s_w'}}

    grid = Grid(ds, coords=coords, periodic=[])

    print('making pm/pn metrics')
    ds['pm_v'] = grid.interp(ds.pm, 'Y')
    ds['pn_u'] = grid.interp(ds.pn, 'X')
    ds['pm_u'] = grid.interp(ds.pm, 'X')
    ds['pn_v'] = grid.interp(ds.pn, 'Y')
    ds['pm_psi'] = grid.interp(grid.interp(ds.pm, 'Y'),  'X') # at psi points (eta_v, xi_u) 
    ds['pn_psi'] = grid.interp(grid.interp(ds.pn, 'X'),  'Y') # at psi points (eta_v, xi_u)
    print('making dx/dy')
    ds['dx'] = 1/ds.pm
    ds['dx_u'] = 1/ds.pm_u
    ds['dx_v'] = 1/ds.pm_v
    ds['dx_psi'] = 1/ds.pm_psi

    ds['dy'] = 1/ds.pn
    ds['dy_u'] = 1/ds.pn_u
    ds['dy_v'] = 1/ds.pn_v
    ds['dy_psi'] = 1/ds.pn_psi

    ds['dz'] = grid.diff(ds.z_w0, 'Z', boundary='fill')
    ds['dz_w'] = grid.diff(ds.z_rho0, 'Z', boundary='fill')
    ds['dz_u'] = grid.interp(ds.dz, 'X')
    ds['dz_w_u'] = grid.interp(ds.dz_w, 'X')
    ds['dz_v'] = grid.interp(ds.dz, 'Y')
    ds['dz_w_v'] = grid.interp(ds.dz_w, 'Y')

    ds['dA'] = ds.dx * ds.dy

    metrics = {
        ('X',): ['dx', 'dx_u', 'dx_v', 'dx_psi'], # X distances
        ('Y',): ['dy', 'dy_u', 'dy_v', 'dy_psi'], # Y distances
        ('Z',): ['dz', 'dz_u', 'dz_v', 'dz_w', 'dz_w_u', 'dz_w_v'], # Z distances
        ('X', 'Y'): ['dA'] # Areas
    }
    grid = Grid(ds, coords=coords, metrics=metrics, periodic=[])

    return ds,grid



def add_zeros_to_4(date):
    if date<10:
        to_add = '000'
    elif date>9 & date<100:
        to_add = '00'
    elif date>99 & date < 1000:
        to_add = '0'
    else: 
        to_add = ''
    return to_add

def generateFileList(FilePath,prefix,datelist):
    filelist=[FilePath+prefix+add_zeros_to_4(datelist[0])+str(datelist[0])+'.nc']
    for dates in datelist[1:]:
        filenameToAppend=FilePath+prefix+add_zeros_to_4(dates)+str(dates)+'.nc'
        filelist.append(filenameToAppend)
    return filelist


def vorticity(u, v, grd):
    """
    compute the relative vorticity
    https://github.com/ESMG/pyroms/blob/python3/pyroms_toolbox/pyroms_toolbox/vorticity.py
    e.g. vorticity(ds.u.isel(s_rho=-1,ocean_time=1),ds.v.isel(s_rho=-1,ocean_time=1),grd)
    """

    dx = 1./grd.pm.values
    dy = 1./grd.pn.values

    #dx, dy at psi point
    dx = 0.5 * (dx[1:,:] + dx[:-1,:])
    dy = 0.5 * (dy[:,1:] + dy[:,:-1])

    vorticity = np.zeros(grd.mask_psi.shape)

    vorticity = 2 * (v[:,1:] - v[:,:-1]) / (dx[:,1:] + dx[:,:-1]) \
                - 2 * (u[1:,:] - u[:-1,:]) / (dy[1:,:] + dy[:-1,:])

    vorticity = np.ma.masked_where(grd.mask_psi == 0, vorticity)

def N2(rho, z, rho_0=1000.0):
    '''
    Return the stratification frequency
    
    Parameters
    ----------
    rho : array_like
        density [kg/m^3]
    z : array_like
        depths [m] (positive upward)
    
    Returns
    -------
    N2 : array_like
        Stratification frequency [1/s], where the vertical dimension (the
        first dimension) is one less than the input arrays.    
    '''
    rho = np.asarray(rho)
    z = np.asarray(z)
    assert rho.shape == z.shape, 'rho and z must have the same shape.'
    r_z = np.diff(rho, axis=0) / np.diff(z, axis=0)
    return -(9.8 / rho_0) * r_z


# Load Data

In [4]:
# load data file

grd = xr.open_dataset('/g/data/jk72/deg581/se-qld-setup/data/proc/seqld_1km_v1.7_grd.nc')

FilePath='/g/data/jk72/deg581/seqom/seqom_v1.7_2012_continue2/' #

prefix='roms_his_'
timeRange = [17,18]
datelist = np.array(range(timeRange[0],timeRange[1],1))


fl=generateFileList(FilePath,prefix,datelist)
print(fl)

# ds=loadOverlappedNetcdfFileList(filelist=fl,overlapDays=7)

ds = xr.open_mfdataset(fl,chunks = {'ocean_time':1}, data_vars='minimal', compat='override',coords='minimal',parallel='False',join='right')

print(ds.nbytes/1e9,'G')

ds = ds.drop_vars(['ubar_eastward','vbar_northward','rho','shflux','ssflux','sustr','svstr'])
print(ds.nbytes/1e9,'G')
ds

ds = ds.assign_coords({"lon_rho": grd.lon_rho})
ds = ds.assign_coords({"lat_rho": grd.lat_rho})

weights_area = (1/ds.pm)*(1/ds.pn)
weights_area.name = "weights"

print('making vertical coordinates')
Zo_rho = (ds.hc * ds.s_rho + ds.Cs_r * ds.h) / (ds.hc + ds.h)
z_rho =  ( ds.h) * Zo_rho
Zo_w = (ds.hc * ds.s_w + ds.Cs_w * ds.h) / (ds.hc + ds.h)
z_w = Zo_w * ( + ds.h) 
    
ds.coords['z_w0'] = z_w.where(ds.mask_rho, 0).transpose('s_w', 'eta_rho', 'xi_rho')
ds.coords['z_rho0'] = z_rho.where(ds.mask_rho, 0).transpose('s_rho', 'eta_rho', 'xi_rho')

ds['dz'] = (('s_rho', 'eta_rho', 'xi_rho'),np.diff(ds.z_w0,axis=0))


ds, grid = ROMSmetricsAndGrid(ds)

dz_top = ds.z_w0.isel(s_w=-1) - ds.z_rho0.isel(s_rho=-1)
dz_bottom = ds.z_rho0.isel(s_rho=0) - ds.z_w0.isel(s_w=0)
ds['dz_w'] = grid.diff(ds.z_rho0, 'Z')
ds.dz_w.isel(s_w=-1)[:] = dz_top.values
ds.dz_w.isel(s_w=0)[:] = dz_bottom.values

# ds_17_again = ds

# ds.close()

['/g/data/jk72/deg581/seqom/seqom_v1.7_2012_continue2/roms_his_0017.nc']
39.617066672 G
33.900014732 G
making vertical coordinates
making pm/pn metrics
making dx/dy


In [5]:
ds.load()


<xarray.Dataset>
Dimensions:         (tracer: 2, boundary: 4, s_rho: 31, s_w: 32, Nuser: 1,
                     eta_rho: 720, xi_rho: 735, xi_u: 734, eta_v: 719,
                     ocean_time: 73)
Coordinates: (12/15)
  * s_rho           (s_rho) float64 -0.9839 -0.9516 ... -0.04839 -0.01613
  * s_w             (s_w) float64 -1.0 -0.9677 -0.9355 ... -0.06452 -0.03226 0.0
    x_rho           (eta_rho, xi_rho) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    y_rho           (eta_rho, xi_rho) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    x_u             (eta_rho, xi_u) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    y_u             (eta_rho, xi_u) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    ...              ...
    y_psi           (eta_v, xi_u) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
  * ocean_time      (ocean_time) datetime64[ns] 2016-01-02 ... 2016-12-27
    lon_rho         (eta_rho, xi_rho) float64 151.5 151.5 151.5 ... 158.8 158.8
    lat_rho         (eta_rho, xi_rho) float64 -31.2 -31.2 ... -24.01 -24.01
    z_w0            (s_w, eta_rho, xi_rho) float64 0.0 0.0 0.0 ... 0.0 0.0 0.0
    z_rho0          (s_rho, eta_rho, xi_rho) float64 0.0 0.0 ... -1.115 -1.116
Dimensions without coordinates: tracer, boundary, Nuser, eta_rho, xi_rho, xi_u,
                                eta_v
Data variables: (12/117)
    ntimes          int32 1051200
    ndtfast         int32 20
    dt              float64 30.0
    dtfast          float64 1.5
    dstart          datetime64[ns] 2000-01-01
    nHIS            int32 14400
    ...              ...
    dz_w            (s_w, eta_rho, xi_rho) float64 0.0 0.0 ... -2.54e+03
    dz_u            (s_rho, eta_rho, xi_u) float64 0.0 0.0 0.0 ... 2.875 2.881
    dz_w_u          (s_w, eta_rho, xi_u) float64 0.0 0.0 0.0 ... 1.114 1.116
    dz_v            (s_rho, eta_v, xi_rho) float64 0.0 0.0 0.0 ... 2.877 2.881
    dz_w_v          (s_w, eta_v, xi_rho) float64 0.0 0.0 0.0 ... 1.115 1.116
    dA              (eta_rho, xi_rho) float64 1.058e+06 1.058e+06 ... 1.129e+06
Attributes: (12/35)
    file:              roms_his_0017.nc
    format:            netCDF-3 64bit offset file
    Conventions:       CF-1.4, SGRID-0.3
    type:              ROMS/TOMS history file
    title:             South-east Queensland, 1/100 (900m) degree resolution
    var_info:          ROMS/External/varinfo.yaml
    ...                ...
    compiler_command:  /apps/openmpi/4.0.2/bin/mpif90
    compiler_flags:    -fp-model precise -heap-arrays -ip -O3 -traceback -che...
    tiling:            024x020
    history:           ROMS/TOMS, Version 4.2, Saturday - January 10, 2026 - ...
    ana_file:          ROMS/Functionals/ana_btflux.h
    CPP_options:       SEQLD, ANA_BSFLUX, ANA_BTFLUX, ASSUMED_SHAPE, AVERAGES...

## define time periods

In [6]:
winter_period=slice(30,40)
summer_period=slice(72,73)

winter_period_snapshot=slice(38,39)
summer_period_snapshot=slice(72,73)

# Do analyses

In [7]:
# FIRST ROMS

# potential vorticity
rel_vort = grid.derivative(ds.v, 'X') - grid.derivative(ds.u, 'Y')
rel_vort_norm = rel_vort / grid.interp(grid.interp(ds.f,'X'),'Y')

strain = np.sqrt( (grid.derivative(ds.u,'X') - grid.derivative(ds.v,'Y'))**2 + (grid.interp(grid.interp(grid.derivative(ds.u,'Y'),'X'),'Y') + grid.interp(grid.interp(grid.derivative(ds.v,'X'),'X'),'Y') )**2 )
strain_norm = strain / ds.f

div = grid.derivative(ds.u, 'X') + grid.derivative(ds.v, 'Y')
div_norm = div / ds.f

# #skew

# # skew_period = slice(25,40)
# skew_period = slice(0,73)


rel_vort_norm.load()
strain_norm.load()
div_norm.load()

print('calc vetical velocity variance')
w_var = ((ds.w - ds.w.mean(dim='ocean_time'))**2).mean(dim='ocean_time')

w_var.load()

calc vetical velocity variance


<xarray.DataArray 'w' (s_w: 32, eta_rho: 720, xi_rho: 735)>
array([[[           nan,            nan,            nan, ...,
         8.58672702e-17, 2.68897703e-16, 2.68897703e-16],
        [           nan,            nan,            nan, ...,
         8.58672702e-17, 2.68897703e-16, 2.68897703e-16],
        [           nan,            nan,            nan, ...,
         1.23869873e-16, 7.22392420e-16, 7.22392420e-16],
        ...,
        [           nan,            nan,            nan, ...,
         1.49381123e-07, 2.52595896e-07, 2.52595896e-07],
        [           nan,            nan,            nan, ...,
         4.66487506e-07, 3.70775837e-07, 3.70775837e-07],
        [           nan,            nan,            nan, ...,
         4.66487506e-07, 3.70775837e-07, 3.70775837e-07]],

       [[           nan,            nan,            nan, ...,
         8.88854702e-05, 1.42281395e-04, 1.42281395e-04],
        [           nan,            nan,            nan, ...,
         8.88854702e-05, 1.42281395e-04, 1.42281395e-04],
        [           nan,            nan,            nan, ...,
         2.50847024e-05, 7.60463881e-05, 7.60463881e-05],
...
        [           nan,            nan,            nan, ...,
         3.85075580e-08, 7.77740041e-08, 7.77740041e-08],
        [           nan,            nan,            nan, ...,
         6.06705370e-08, 2.15003780e-07, 2.15003780e-07],
        [           nan,            nan,            nan, ...,
         6.06705370e-08, 2.15003780e-07, 2.15003780e-07]],

       [[           nan,            nan,            nan, ...,
         5.31569753e-12, 3.14544814e-11, 3.14544814e-11],
        [           nan,            nan,            nan, ...,
         5.31569753e-12, 3.14544814e-11, 3.14544814e-11],
        [           nan,            nan,            nan, ...,
         1.05329192e-11, 5.58086181e-11, 5.58086181e-11],
        ...,
        [           nan,            nan,            nan, ...,
         5.89875154e-13, 5.95470776e-13, 5.95470776e-13],
        [           nan,            nan,            nan, ...,
         1.01583878e-12, 7.91184067e-13, 7.91184067e-13],
        [           nan,            nan,            nan, ...,
         1.01583878e-12, 7.91184067e-13, 7.91184067e-13]]], dtype=float32)
Coordinates:
  * s_w      (s_w) float64 -1.0 -0.9677 -0.9355 ... -0.06452 -0.03226 0.0
    x_rho    (eta_rho, xi_rho) float64 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    y_rho    (eta_rho, xi_rho) float64 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    lon_rho  (eta_rho, xi_rho) float64 151.5 151.5 151.5 ... 158.8 158.8 158.8
    lat_rho  (eta_rho, xi_rho) float64 -31.2 -31.2 -31.2 ... -24.01 -24.01
    z_w0     (s_w, eta_rho, xi_rho) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Dimensions without coordinates: eta_rho, xi_rho

In [8]:
import gsw

# Shapes
s_rho = ds.s_rho.size
ot = ds.ocean_time.size
eta = ds.eta_rho.size
xi = ds.xi_rho.size

# Get full 4D fields
SP = ds.salt.values      # shape: (s_rho, doy, eta, xi)
pt = ds.temp.values      # potential temperature
z = ds.z_rho0.values         # depth in m (negative)

# Convert depth to pressure
p = gsw.p_from_z(z, lat=-67)        # shape: (s_rho, doy, eta, xi)

# Convert to Absolute Salinity
SA = gsw.SA_from_SP(SP, p, lon=60, lat=-67)  # shape: same as SP

# Convert to Conservative Temperature
CT = gsw.CT_from_pt(SA, pt)

# Calculate sigma0 directly
sigma_0 = gsw.sigma0(SA, CT)

# Assign to dataset

sigma0 = np.transpose(sigma_0, (1, 0, 2, 3))
ds['sigma0'] = (('s_rho', 'ocean_time', 'eta_rho', 'xi_rho'), sigma0)

In [9]:
# calculate density gradient

rho_sfc = ds.sigma0.isel(s_rho=-1)

drho_dx = grid.derivative(rho_sfc, 'X')
drho_dy = grid.derivative(rho_sfc, 'Y')

drho_dx_rho = grid.interp(drho_dx, 'X')
drho_dy_rho = grid.interp(drho_dy, 'Y')

grad_rho = np.sqrt(drho_dx_rho**2 + drho_dy_rho**2)
ds['grad_rho_surf']=grad_rho


In [10]:
# calculate isopycnal tilt

drhodx = grid.derivative(ds.sigma0,'X')

ds['drhodx'] = grid.interp(drhodx,'X')


In [11]:
# calculate REAL isopycnal tilt.
# Step 3: horizontal magnitude of slope (2-norm of isopycnal tilt)
# S = sqrt((dz/dx)**2 + (dz/dy)**2)
#    = sqrt((drho/dx / drho/dz)**2 + (drho/dy / drho/dz)**2)
#    = sqrt((drho/dx)**2 + (drho/dy)**2) / |drho/dz|

# Step 4: compact form using horizontal gradient
# S = |∇_h z_rho| = |∇_h rho| / |∂rho/∂z|
# Optionally, using buoyancy:
# S = |∇_h b| / N^2

drhodx = grid.derivative(ds.sigma0,'X')
drhody = grid.derivative(ds.sigma0,'Y')
drhodz = grid.derivative(ds.sigma0,'Z')

ds['S'] = (grid.interp(drhodx,'X')**2 + grid.interp(drhody,'Y')**2 )**0.5 / np.abs(grid.interp(drhodz,'Z'))




In [ ]:
ds.S.load()

# begin plotting

In [ ]:
import pandas as pd # for labelling purposes


period_snapshot_1 = slice(38,39)
period_snapshot_2 = slice(56,57)
period_snapshot_3 = slice(72,73)


In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.ticker as ticker

In [ ]:
# define region boxes

lon_min,lon_max = 153, 154
lat_min, lat_max= -24.3, -26.6

In [ ]:
ds.h.plot(x='lon_rho',y='lat_rho',vmax=75)
plt.plot(ds.lon_rho.isel(eta_rho=620),ds.lat_rho.isel(eta_rho=620))

In [ ]:
# interp the fields
# rel_vort_norm
rv_rho = 0.25 * (
    rel_vort_norm.isel(eta_v=slice(0, -1), xi_u=slice(0, -1)) +
    rel_vort_norm.isel(eta_v=slice(1, None), xi_u=slice(0, -1)) +
    rel_vort_norm.isel(eta_v=slice(0, -1), xi_u=slice(1, None)) +
    rel_vort_norm.isel(eta_v=slice(1, None), xi_u=slice(1, None))
)

In [ ]:
rv_full = xr.full_like(ds.w[:,0:-1,:,:], np.nan)
rv_full[:,:,1:-1, 1:-1] = rv_rho

In [ ]:
import matplotlib.ticker as mticker
from cartopy.mpl.ticker import LongitudeFormatter

In [ ]:
fig = plt.figure(figsize=[8,9])
gs = gridspec.GridSpec(nrows=4,ncols=4,wspace=0.02, hspace=0.08,height_ratios=[1, 0.22, 0.35,0.35])

choose_depth=0
transect_choose = 620
transect_lon_min,transect_lon_max = 153.35,153.9

# OUR MODEL

##### Ro number
ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
# im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.mean(dim='ocean_time').isel(s_rho=-1),cmap='RdBu_r',vmin=-2,vmax=2)
# co = ax.contour(grd.lon_rho,grd.lat_rho,rv_full.mean(dim='ocean_time').isel(s_w=-1),colors='k',linewidths=0.5,levels=[-1,1])

# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')

ax.plot(ds.lon_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        ds.lat_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        color='C0',linewidth=1)

ax.set_extent([lon_min,lon_max,lat_min,lat_max])
gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.xlocator = mticker.MultipleLocator(0.1)

gl.xlabel_style = {'rotation': 45}

In [ ]:


# snapshots of submeso diagnostics

plt.cla()
plt.clf()
fig = plt.figure(figsize=[8,9])
ax = None

gs = gridspec.GridSpec(nrows=4,ncols=4,wspace=0.02, hspace=0.16,height_ratios=[1, 0.22, 0.35,0.35])

choose_depth=0
transect_choose = 620
transect_lon_min,transect_lon_max = 153.35,153.9

# OUR MODEL

##### Ro number
ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.mean(dim='ocean_time').isel(s_rho=-1),cmap='RdBu_r',vmin=-2,vmax=2)
co = ax.contour(grd.lon_rho,grd.lat_rho,rv_full.mean(dim='ocean_time').isel(s_w=-1),colors='k',linewidths=0.5,levels=[-1,1])

# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')

ax.plot(ds.lon_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        ds.lat_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        color='k',linewidth=1.5)

ax.set_extent([lon_min,lon_max,lat_min,lat_max])
gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.xlocator = mticker.MultipleLocator(0.1)
gl.xlabel_style = {'rotation': 45}
ax.text(0.019, 0.98, 'a', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
# ax.text(0.019, 0.91, 'Ro', transform=ax.transAxes,fontsize=22, fontweight='normal', va='top')

# central_lat = -27.5  # adjust for your map
# scale_length_km = 30
# deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
# scale_length_deg = scale_length_km * deg_per_km
# lon0 = 152.6  # scale bar position
# lat0 = -30.3
# # Add scale bar
# ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
#         transform=ccrs.PlateCarree(), color='black', lw=4)
# ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
#         transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

# tt = ds.ocean_time.isel(ocean_time=period_snapshot_1).values[0]
# label = pd.to_datetime(tt).strftime('%B-%d')
# ax.set_title(label,fontsize=20)

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\zeta/f$')#,fontsize=14)

##### divergence

ax=fig.add_subplot(gs[0,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).mean(dim='ocean_time').squeeze(),cmap='cmo.curl',vmin=-1,vmax=1)
# ax.contour(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = False
gl.xlocator = mticker.MultipleLocator(0.1)
ax.text(0.019, 0.98, 'b', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
gl.xlabel_style = {'rotation': 45}

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\delta/f$')#,fontsize=14)


##### strain
ax=fig.add_subplot(gs[0,2],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,strain_norm.isel(s_rho=-1).mean(dim='ocean_time').squeeze(),cmap='RdBu_r',vmin=-1,vmax=1)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = False
ax.text(0.019, 0.98, 'c', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
gl.xlocator = mticker.MultipleLocator(0.1)
gl.xlabel_style = {'rotation': 45}

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$|\mathbf{S}|/f$')#,fontsize=14)


##### <grad(rho)>
ax=fig.add_subplot(gs[0,3],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,ds.grad_rho_surf.mean(dim='ocean_time'),cmap='cmo.rain_r',vmin=0,vmax=0.1e-3)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False
gl.bottom_labels = True
ax.text(0.019, 0.98, 'd', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
gl.xlocator = mticker.MultipleLocator(0.1)
gl.xlabel_style = {'rotation': 45}

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\nabla_h \rho$')#,fontsize=14)
cbar.formatter = ticker.ScalarFormatter(useMathText=True)
cbar.formatter.set_powerlimits((-2, 2))
cbar.update_ticks()
# vertical sections

ax=fig.add_subplot(gs[2,:])
im = ax.pcolormesh(ds.lon_rho.isel(eta_rho=transect_choose),ds.z_rho0.isel(eta_rho=transect_choose),grid.interp(w_var,'Z').isel(eta_rho=transect_choose),norm=LogNorm(vmin=1e-9,vmax=1e-5))
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((transect_lon_min,transect_lon_max))
ax.set_ylim((-750,0))
ax.set_ylabel('Depth (m)')
ax.set_xticklabels('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: rf"{x:.1f}$^\circ$E"))
ax.text(0.019, 0.98, 'e', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
cax = inset_axes(ax,
                width="45%",  # width = 10% of parent_bbox width
                height="4%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.05,0.22, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_title(r"$\langle w'^2 \rangle$  ($\mathrm{m}^2/\mathrm{s}^2$)")#,fontsize=14)

ax=fig.add_subplot(gs[3,:])
im = ax.pcolormesh(ds.lon_rho.isel(eta_rho=transect_choose),ds.z_rho0.isel(eta_rho=transect_choose),ds.drhodx.isel(eta_rho=transect_choose).mean(dim='ocean_time'))
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((transect_lon_min,transect_lon_max))
ax.set_ylim((-750,0))
ax.set_ylabel('Depth (m)')
ax.set_xlabel('Longitude')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: rf"{x:.1f}$^\circ$E"))
ax.text(0.019, 0.98, 'f', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
cax = inset_axes(ax,
                width="45%",  # width = 10% of parent_bbox width
                height="4%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.05,0.22, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
# cax.set_xlabel(r"$\frac{\partial \rho}{\partial x}$  ($\mathrm{kg}/\mathrm{m}^4$)")#,fontsize=14)
cax.set_title(r"Isopycnal slope ($\mathrm{kg}/\mathrm{m}^4$)")#,fontsize=14)

In [ ]:


# snapshots of submeso diagnostics

plt.cla()
plt.clf()
fig = plt.figure(figsize=[8,9])
ax = None

gs = gridspec.GridSpec(nrows=4,ncols=4,wspace=0.02, hspace=0.17,height_ratios=[1, 0.22, 0.35,0.35])

choose_depth=0
transect_choose = 620
transect_lon_min,transect_lon_max = 153.35,153.9

# OUR MODEL

##### Ro number
ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.mean(dim='ocean_time').isel(s_rho=-1),cmap='RdBu_r',vmin=-2,vmax=2)
co = ax.contour(grd.lon_rho,grd.lat_rho,rv_full.mean(dim='ocean_time').isel(s_w=-1),colors='k',linewidths=0.5,levels=[-1,1])

# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')

ax.plot(ds.lon_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        ds.lat_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        color='C0',linewidth=1.5)

ax.set_extent([lon_min,lon_max,lat_min,lat_max])
gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.xlocator = mticker.MultipleLocator(0.1)
gl.xlabel_style = {'rotation': 45}
ax.text(0.042, 0.11, 'a', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.8))
# ax.text(0.019, 0.91, 'Ro', transform=ax.transAxes,fontsize=22, fontweight='normal', va='top')

# central_lat = -27.5  # adjust for your map
# scale_length_km = 30
# deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
# scale_length_deg = scale_length_km * deg_per_km
# lon0 = 152.6  # scale bar position
# lat0 = -30.3
# # Add scale bar
# ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
#         transform=ccrs.PlateCarree(), color='black', lw=4)
# ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
#         transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

# tt = ds.ocean_time.isel(ocean_time=period_snapshot_1).values[0]
# label = pd.to_datetime(tt).strftime('%B-%d')
# ax.set_title(label,fontsize=20)

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\zeta/f$')#,fontsize=14)

##### divergence

ax=fig.add_subplot(gs[0,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).mean(dim='ocean_time').squeeze(),cmap='cmo.curl',vmin=-1,vmax=1)
# ax.contour(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = False
gl.xlocator = mticker.MultipleLocator(0.1)
ax.text(0.042, 0.11, 'b', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.8))
gl.xlabel_style = {'rotation': 45}

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\delta/f$')#,fontsize=14)


##### strain
ax=fig.add_subplot(gs[0,2],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,strain_norm.isel(s_rho=-1).mean(dim='ocean_time').squeeze(),cmap='RdBu_r',vmin=-1,vmax=1)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = False
ax.text(0.042, 0.11, 'c', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.8))
gl.xlocator = mticker.MultipleLocator(0.1)
gl.xlabel_style = {'rotation': 45}

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$|\mathbf{S}|/f$')#,fontsize=14)


##### <grad(rho)>
ax=fig.add_subplot(gs[0,3],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,ds.grad_rho_surf.mean(dim='ocean_time'),cmap='cmo.rain_r',vmin=0,vmax=0.1e-3)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False
gl.bottom_labels = True
ax.text(0.042, 0.11, 'd', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.8))
gl.xlocator = mticker.MultipleLocator(0.1)
gl.xlabel_style = {'rotation': 45}

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\nabla_h \rho$')#,fontsize=14)
cbar.formatter = ticker.ScalarFormatter(useMathText=True)
cbar.formatter.set_powerlimits((-2, 2))
cbar.update_ticks()
# vertical sections

ax=fig.add_subplot(gs[2,:])
im = ax.pcolormesh(ds.lon_rho.isel(eta_rho=transect_choose),ds.z_rho0.isel(eta_rho=transect_choose),grid.interp(w_var,'Z').isel(eta_rho=transect_choose),norm=LogNorm(vmin=1e-9,vmax=1e-5))
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((transect_lon_min,transect_lon_max))
ax.set_ylim((-750,0))
ax.set_ylabel('Depth (m)')
ax.set_xticklabels('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: rf"{x:.1f}$^\circ$E"))
ax.text(0.013, 0.94, 'e', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
cax = inset_axes(ax,
                width="45%",  # width = 10% of parent_bbox width
                height="4%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.05,0.23, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_title(r"$ \overline{w'^2}$  ($\mathrm{m}^2/\mathrm{s}^2$)")#,fontsize=14)

ax=fig.add_subplot(gs[3,:])
im = ax.pcolormesh(ds.lon_rho.isel(eta_rho=transect_choose),ds.z_rho0.isel(eta_rho=transect_choose),ds.S.isel(eta_rho=transect_choose).mean(dim='ocean_time'),vmax=0.1)
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((transect_lon_min,transect_lon_max))
ax.set_ylim((-750,0))
ax.set_ylabel('Depth (m)')
ax.set_xlabel('Longitude')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: rf"{x:.1f}$^\circ$E"))
ax.text(0.013, 0.94, 'f', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
cax = inset_axes(ax,
                width="45%",  # width = 10% of parent_bbox width
                height="4%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.05,0.22, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
# cax.set_xlabel(r"$\frac{\partial \rho}{\partial x}$  ($\mathrm{kg}/\mathrm{m}^4$)")#,fontsize=14)
cax.set_title(r"Isopycnal slope, $s$")#,fontsize=14)

In [ ]:
ds.S.isel(eta_rho=transect_choose).mean(dim='ocean_time').plot(vmax=0.1)

In [ ]:
ds.S.isel(eta_rho=transect_choose).mean(dim='ocean_time').isel(xi_rho=400,s_rho=15)

In [ ]:
# define region boxes

lon_min,lon_max = 153.3, 154
lat_min, lat_max= -26.9, -29

In [ ]:
ds.h.plot(x='lon_rho',y='lat_rho',vmax=75)
plt.plot(ds.lon_rho.isel(eta_rho=240),ds.lat_rho.isel(eta_rho=240))

In [ ]:


# snapshots of submeso diagnostics

plt.cla()
plt.clf()
fig = plt.figure(figsize=[7,9])
ax = None

gs = gridspec.GridSpec(nrows=4,ncols=4,wspace=0.02, hspace=0.08,height_ratios=[1, 0.22, 0.35,0.35])

choose_depth=0
transect_choose = 240
transect_lon_min,transect_lon_max = 153.6,153.9

# OUR MODEL

##### Ro number
ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.mean(dim='ocean_time').isel(s_rho=-1),cmap='RdBu_r',vmin=-2,vmax=2)
# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')

ax.plot(ds.lon_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        ds.lat_rho.isel(eta_rho=transect_choose).where((ds.lon_rho.isel(eta_rho=transect_choose)>transect_lon_min) & (ds.lon_rho.isel(eta_rho=transect_choose)<transect_lon_max),drop=True),
        color='C1',linewidth=1)

ax.set_extent([lon_min,lon_max,lat_min,lat_max])
gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True

ax.text(0.019, 0.98, 'a', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
# ax.text(0.019, 0.91, 'Ro', transform=ax.transAxes,fontsize=22, fontweight='normal', va='top')

# central_lat = -27.5  # adjust for your map
# scale_length_km = 30
# deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
# scale_length_deg = scale_length_km * deg_per_km
# lon0 = 152.6  # scale bar position
# lat0 = -30.3
# # Add scale bar
# ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
#         transform=ccrs.PlateCarree(), color='black', lw=4)
# ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
#         transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

# tt = ds.ocean_time.isel(ocean_time=period_snapshot_1).values[0]
# label = pd.to_datetime(tt).strftime('%B-%d')
# ax.set_title(label,fontsize=20)

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.15, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\zeta/f$')#,fontsize=14)

##### divergence

ax=fig.add_subplot(gs[0,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).mean(dim='ocean_time').squeeze(),cmap='cmo.curl',vmin=-1,vmax=1)
# ax.contour(grd.lon_rho,grd.lat_rho,div_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = False

ax.text(0.019, 0.98, 'b', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))

cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.15, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\delta/f$')#,fontsize=14)


##### strain
ax=fig.add_subplot(gs[0,2],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,strain_norm.isel(s_rho=-1).mean(dim='ocean_time').squeeze(),cmap='RdBu_r',vmin=-1,vmax=1)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = False
ax.text(0.019, 0.98, 'c', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))


cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.15, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$|\mathbf{S}|/f$')#,fontsize=14)


##### <grad(rho)>
ax=fig.add_subplot(gs[0,3],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,ds.grad_rho_surf.mean(dim='ocean_time'),cmap='cmo.rain_r',vmin=0,vmax=0.1e-3)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.contour(ds.lon_rho,ds.lat_rho,ds.h,levels = [100,500,1000,1500,2000],linewidths=1,colors='0.8')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False
gl.bottom_labels = True
ax.text(0.019, 0.98, 'd', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))


cax = inset_axes(ax,
                width="75%",  # width = 10% of parent_bbox width
                height="3%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.1,-0.15, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\nabla_h \rho$')#,fontsize=14)
cbar.formatter = ticker.ScalarFormatter(useMathText=True)
cbar.formatter.set_powerlimits((-2, 2))
cbar.update_ticks()
# vertical sections

ax=fig.add_subplot(gs[2,:])
im = ax.pcolormesh(ds.lon_rho.isel(eta_rho=transect_choose),ds.z_rho0.isel(eta_rho=transect_choose),grid.interp(w_var,'Z').isel(eta_rho=transect_choose),norm=LogNorm(vmin=1e-9,vmax=1e-5))
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((transect_lon_min,transect_lon_max))
ax.set_ylim((-750,0))
ax.set_ylabel('Depth (m)')
ax.set_xticklabels('')
ax.text(0.019, 0.98, 'e', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
cax = inset_axes(ax,
                width="45%",  # width = 10% of parent_bbox width
                height="4%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.05,0.21, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_title(r"$\langle w'^2 \rangle$  ($\mathrm{m}^2/\mathrm{s}^2$)")#,fontsize=14)

ax=fig.add_subplot(gs[3,:])
im = ax.pcolormesh(ds.lon_rho.isel(eta_rho=transect_choose),ds.z_rho0.isel(eta_rho=transect_choose),ds.drhodx.isel(eta_rho=transect_choose).mean(dim='ocean_time'))
# ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_xlim((transect_lon_min,transect_lon_max))
ax.set_ylim((-750,0))
ax.set_ylabel('Depth (m)')
ax.set_xlabel('Longitude')
ax.text(0.019, 0.98, 'f', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
cax = inset_axes(ax,
                width="45%",  # width = 10% of parent_bbox width
                height="4%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(.05,0.21, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
# cax.set_xlabel(r"$\frac{\partial \rho}{\partial x}$  ($\mathrm{kg}/\mathrm{m}^4$)")#,fontsize=14)
cax.set_title(r"Isopycnal slope ($\mathrm{kg}/\mathrm{m}^4$)")#,fontsize=14)

In [ ]:
STOP

# now do same over top 100m

In [ ]:
# ROMS
top_mask = ds.z_w0>=-100

w_var_topX = ((w_var * ds.dz_w).where(top_mask).sum(dim='s_w', skipna=True) /  ds.dz_w.where(top_mask).sum(dim='s_w', skipna=True))

# BRAN
dz_bran = ds_w.sw_edges_ocean.diff(dim='sw_edges_ocean')
top_mask_bran =ds_w.sw_ocean < 100

w_var_topX_bran = ((w_var_bran * dz_bran).where(top_mask_bran).sum(dim='sw_ocean', skipna=True) / dz_bran.where(top_mask_bran).sum(dim='sw_ocean', skipna=True))

In [ ]:
np.nansum(dz_bran_3d*top_mask_bran_3d,axis=0)

In [ ]:
w_var_bran[0:14].mean(dim='sw_ocean').plot(vmax=1e-7)

In [ ]:
w_var_topX_bran.plot(vmax=1e-7)

In [ ]:
w_var_topX.plot(vmin=0,vmax=1e-6)

In [ ]:
from matplotlib.colors import LogNorm

In [ ]:
# snapshots of submeso diagnostics

plt.cla()
plt.clf()
fig = plt.figure(figsize=[12,9])
ax = None

gs = gridspec.GridSpec(nrows=2,ncols=3,wspace=0.02, hspace=0.02)

choose_depth=0
norm_logwvar=LogNorm(vmin=1e-10,vmax=1e-5)

# OUR MODEL

##### Ro number
ax=fig.add_subplot(gs[0,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,rel_vort_norm.mean(dim='ocean_time').isel(s_rho=-1),cmap='RdBu_r',vmin=-2,vmax=2)
# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])
gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = False

ax.text(0.02, 0.98, 'a', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
# ax.text(0.019, 0.91, 'Ro', transform=ax.transAxes,fontsize=22, fontweight='normal', va='top')

central_lat = -27.5  # adjust for your map
scale_length_km = 30
deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
scale_length_deg = scale_length_km * deg_per_km
lon0 = 152.6  # scale bar position
lat0 = -30.3
# Add scale bar
ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
        transform=ccrs.PlateCarree(), color='black', lw=4)
ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
        transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

# tt = ds.ocean_time.isel(ocean_time=period_snapshot_1).values[0]
# label = pd.to_datetime(tt).strftime('%B-%d')
# ax.set_title(label,fontsize=20)

##### strain
ax=fig.add_subplot(gs[0,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,strain_norm.isel(s_rho=-1).mean(dim='ocean_time').squeeze(),cmap='RdBu_r',vmin=-1,vmax=1)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = False
gl.left_labels = False

ax.text(0.02, 0.98, 'b', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))


##### <w'>
ax=fig.add_subplot(gs[0,2],projection=ccrs.PlateCarree())
im = ax.pcolormesh(grd.lon_rho,grd.lat_rho,w_var_topX,norm=norm_logwvar)#,cmap='cmo.rain_r',vmin=0,vmax=0.4e-3)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False
gl.bottom_labels = False
ax.text(0.02, 0.98, 'c', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))

# BRAN MODEL

##### Ro number
ax=fig.add_subplot(gs[1,0],projection=ccrs.PlateCarree())
im = ax.pcolormesh(lon,lat,zeta_norm.mean(dim='Time').isel(st_ocean=0),cmap='RdBu_r',vmin=-2,vmax=2)
# ax.contour(grd.lon_psi,grd.lat_psi,rel_vort_norm.isel(s_rho=-1).isel(ocean_time=winter_period_snapshot).squeeze(),colors='k',linewidths=.5)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])
gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = True

ax.text(0.02, 0.98, 'd', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))
# ax.text(0.019, 0.91, 'Ro', transform=ax.transAxes,fontsize=22, fontweight='normal', va='top')

central_lat = -27.5  # adjust for your map
scale_length_km = 30
deg_per_km = 1 / (111.32 * np.cos(np.radians(central_lat)))
scale_length_deg = scale_length_km * deg_per_km
lon0 = 152.6  # scale bar position
lat0 = -30.3
# Add scale bar
ax.plot([lon0, lon0 + scale_length_deg], [lat0, lat0],
        transform=ccrs.PlateCarree(), color='black', lw=4)
ax.text(lon0 + scale_length_deg / 2, lat0 + 0.002*lat0, '30 km',
        transform=ccrs.PlateCarree(), ha='center', va='top', fontsize=9)

# tt = ds.ocean_time.isel(ocean_time=period_snapshot_1).values[0]
# label = pd.to_datetime(tt).strftime('%B-%d')
# ax.set_title(label,fontsize=20)


cax = inset_axes(ax,
                width="65%",  # width = 10% of parent_bbox width
                height="5%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(0.18,-0.13, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$\zeta/f$')#,fontsize=14)

##### strain
ax=fig.add_subplot(gs[1,1],projection=ccrs.PlateCarree())
im = ax.pcolormesh(lon,lat,strain_bran_norm.mean(dim='Time').isel(st_ocean=0),cmap='RdBu_r',vmin=-1,vmax=1)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.bottom_labels = True
gl.left_labels = False

ax.text(0.02, 0.98, 'e', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))


cax = inset_axes(ax,
                width="65%",  # width = 10% of parent_bbox width
                height="5%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(0.18,-0.13, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r'$|\mathbf{S}|/f$')#,fontsize=14)

##### <w'>
ax=fig.add_subplot(gs[1,2],projection=ccrs.PlateCarree())
im = ax.pcolormesh(lon,lat,w_var_topX_bran,norm=norm_logwvar)#,cmap='RdBu_r',vmin=-1,vmax=1)
ax.contour(grd.lon_rho,grd.lat_rho,grd.mask_rho,levels=[0],colors='k')
ax.set_extent([lon_min,lon_max,lat_min,lat_max])

gl = ax.gridlines(draw_labels=True,
                    color='black', alpha=0.2, linestyle='--')
gl.right_labels = False
gl.top_labels = False
gl.left_labels = False

ax.text(0.02, 0.98, 'f', transform=ax.transAxes,fontsize=22, fontweight='bold', va='top',bbox=dict(boxstyle='round,pad=0.2',facecolor='0.95',edgecolor='black',linewidth=0.8,alpha=0.7))

cax = inset_axes(ax,
                width="65%",  # width = 10% of parent_bbox width
                height="5%",  # height : 50%
                loc='lower left',
                bbox_to_anchor=(0.18,-0.13, 1, 1),
                bbox_transform=ax.transAxes,
                borderpad=0,
                )
cbar = fig.colorbar(im, cax=cax, orientation='horizontal') 
cax.set_xlabel(r"$\langle w'^2 \rangle$ (m$^2$/s$^2$)")#,fontsize=14)

######

